In [ ]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction

# Calculation Without Data: LBCO, CWL

This example shows how to **calculate and plot a diffraction pattern
without any measured data**. Everything the calculation needs — the
crystal structure, the instrument, the peak profile, and the
background — is defined in code, and the pattern is computed over a
calculation range instead of over loaded data points.

This is useful to preview what a candidate structure should look like,
to teach, or to generate a synthetic pattern before any measurement
exists. No data file is downloaded or loaded.

For this example, a constant-wavelength neutron powder experiment for
La0.5Ba0.5CoO3 is used.

## 🛠️ Import Library

In [ ]:
import easydiffraction as ed

## 📦 Define Project

In [ ]:
project = ed.Project(name='lbco_simulation')

## 🧩 Define Structure

### Download CIF file

In [ ]:
structure_path = ed.download_data('struct-lbco', destination='data')

### Add Structure from CIF

In [ ]:
project.structures.add_from_cif_path(structure_path)
project.structures.show_names()

structure = project.structures['lbco']

### Plot Structure

In [ ]:
project.display.structure(struct_name='lbco')

## 🔬 Define Experiment

### Create Experiment Without Data

Instead of loading a measured data file, the 'virtual' experiment is created
directly from its type. With no measured scan present, the pattern is
later computed over the `data_range` defined below.

In [ ]:
project.experiments.create(
    name='sim',
    sample_form='powder',
    beam_mode='constant wavelength',
    radiation_probe='neutron',
)

In [ ]:
experiment = project.experiments['sim']

### Set Instrument

In [ ]:
experiment.instrument.setup_wavelength = 1.494

### Set Peak Profile

In [ ]:
experiment.peak.broad_gauss_u = 0.1
experiment.peak.broad_gauss_v = -0.1
experiment.peak.broad_gauss_w = 0.1
experiment.peak.broad_lorentz_y = 0.1

### Set Background

In [ ]:
experiment.background.create(id='1', position=10, intensity=20)
experiment.background.create(id='2', position=160, intensity=20)

### Set Calculation Range

With no measured data, the x-grid to calculate on comes from the
`data_range` category. It already holds a sensible default window
derived from the instrument, so the experiment is calculable without
any setup. Here we change the default window as an example.

In [ ]:
experiment.data_range.two_theta_min = 10.0
experiment.data_range.two_theta_max = 160.0
experiment.data_range.two_theta_inc = 0.05

### Set Linked Structures

In [ ]:
experiment.linked_structures.create(structure_id='lbco', scale=10.0)

## 🚀 Perform Calculation

### Display Pattern

Plotting the pattern computes the calculated curve over the
`data_range` grid and shows a two-panel view: the calculated curve
with its background on the main panel, plus a Bragg-peaks row. There
is no measured curve or residual, because there is no measurement.

In [ ]:
project.display.pattern(expt_name='sim')

In [ ]:
project.display.pattern(expt_name='sim', x_min=30, x_max=60)

### Modify Parameters and Recalculate

In [ ]:
structure.cell.length_a = 3.6
structure.atom_sites['O'].adp_iso = 1.2
experiment.peak.broad_lorentz_y = 0.9

In [ ]:
project.analysis.calculate()

In [ ]:
project.display.pattern(expt_name='sim', x_min=30, x_max=60)

## 💾 Save Project

In [ ]:
project.save_as(dir_path='projects/ed_27_lbco_simulation')